# MycoMetrics - Notebook 02: Sales & Customer EDA
## Pricing Realization, Customer Concentration, and Volume Elasticity
**Business Focus**: FungiFlow Organics Commercial Revenue Analysis  
**Author**: Data Analytics Team  

### Executive Overview
This notebook analyzes sales orders, price realization across packaging formats (punnets vs crates), customer channel profitability, and tests for volume-price correlation.

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

con = duckdb.connect('../data/warehouse.duckdb')
con.execute("SET search_path = 'analytics_analytics'")

sales = con.execute("""
    select 
        s.order_id,
        c.customer_name,
        c.customer_segment,
        m.mushroom_common_name,
        p.package_format,
        s.quantity_fulfilled_kg,
        s.final_recognized_revenue,
        round(s.final_recognized_revenue / nullif(s.quantity_fulfilled_kg, 0), 2) as realized_asp_per_kg
    from fact_sales s
    inner join dim_customer c on s.customer_id = c.customer_id
    inner join dim_product p on s.product_id = p.product_id
    inner join dim_mushroom m on s.mushroom_id = m.mushroom_id
    where s.order_status in ('DELIVERED', 'COMPLETED')
""").df()

sales.head()

### 1. Price Realization by Packaging Format

In [ ]:
price_summary = sales.groupby('package_format')['realized_asp_per_kg'].agg(['count', 'mean', 'std', 'min', 'max']).round(2)
price_summary

### 2. Volume-Price Correlation (Testing Quantity Discounting Elasticity)

In [ ]:
clean = sales.dropna(subset=['quantity_fulfilled_kg', 'realized_asp_per_kg'])
r, p_pearson = stats.pearsonr(clean['quantity_fulfilled_kg'], clean['realized_asp_per_kg'])
rho, p_spearman = stats.spearmanr(clean['quantity_fulfilled_kg'], clean['realized_asp_per_kg'])

print(f"Pearson Correlation:  r = {r:.3f}, p = {p_pearson:.4f}")
print(f"Spearman Rank Corr:   rho = {rho:.3f}, p = {p_spearman:.4f}")